# Linear Programming Modeling: Be the Decision Maker
**IN2040 · Optimization of Organizational Processes · Tecnológico de Monterrey** (Prof. Vincent Boyer)

*Session 1 · explore during class, about 5 minutes per problem.*

> **How to use this notebook.** Run the **Setup** cell first. Then, for each problem:
> 1. read the **problem**: its data (profits, hours, audiences…) are **fixed**: they describe the situation;
> 2. play with the **explorer**: the sliders are the **decision variables**. Move them and watch the objective
>    and every constraint react (✔ satisfied, ✘ violated). Try to find the best feasible plan, then press
>    **Load the optimal plan** to compare with the solver;
> 3. read the **formulation** (the 4 steps: variables, objective, constraints, non-negativity) and play with
>    the **model with your numbers**: the same sliders, plugged into every equation;
> 4. run the two **PuLP** cells: the same model written **directly** (one line per constraint), then written with
>    **data and sums** (`lpSum`), the way larger models are written.

For each problem, ask yourself:
1. **Which constraints stop you** when you try to improve the objective?
2. Which constraints are **used exactly at their limit** in the optimal plan?
3. How far was **your** plan from the optimum, and why?

In [1]:
# @title ⚙️ Setup: run this cell first (no need to read the code)
import importlib, subprocess, sys
import importlib.metadata
try:                                   # PuLP 4 + its CBC solver (pulp[cbc]); replaces an older PuLP 3
    _old = int(importlib.metadata.version("pulp").split(".")[0]) < 4 or importlib.util.find_spec("cbcbox") is None
except importlib.metadata.PackageNotFoundError:
    _old = True
if _old:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pulp[cbc]>=4"], check=True)
import base64, io, json, traceback
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as W
from IPython.display import display
from scipy.optimize import linprog

PARTS = []      # HTML pieces of the current view, written into one HTML widget (no Output widget: VS Code safe)


def optimum(c, A_ub=None, b_ub=None, A_eq=None, b_eq=None, bounds=None, maximize=False):
    """Optimal solution of the LP (HiGHS), used for the "load the optimal plan" button."""
    c = np.asarray(c, float)
    r = linprog(-c if maximize else c, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq, bounds=bounds, method="highs")
    return r.x + 0.0, (-r.fun if maximize else r.fun)


def banner(ok, text, top=False):
    color, bg = ("#067429", "#eaf9ef") if ok else ("#bd0a0a", "#fff0f0")
    (lambda h: PARTS.insert(0, h) if top else PARTS.append(h))(f'<div style="border:2px solid {color};background:{bg};color:{color};padding:8px 12px;'
                 f'border-radius:8px;font-size:15px;margin:6px 0">{text}</div>')


def table(df):
    PARTS.append(df.to_html(index=False, border=1, escape=False,
                            float_format=lambda v: f"{0.0 if abs(v) < 5e-3 else v:,.2f}")
                 .replace('<table', '<table style="border-collapse:collapse;margin:0 0 8px 0;font-size:13px"', 1))


def show(fig):
    fig.tight_layout()
    buf = io.BytesIO(); fig.savefig(buf, format="png", dpi=100); plt.close(fig)
    PARTS.append(f'<img src="data:image/png;base64,{base64.b64encode(buf.getvalue()).decode()}"/>')


def constraints(rows):
    """rows: (constraint, value of the left-hand side, sense, right-hand side). Shows a ✔/✘ table, returns the violated ones."""
    tol = 1e-4
    ok = [(v <= r + tol) if s == "≤" else (v >= r - tol) if s == "≥" else abs(v - r) <= tol for _, v, s, r in rows]
    table(pd.DataFrame({"Constraint": [n for n, *_ in rows], "Your plan": [v for _, v, _, _ in rows],
                        "": [s for _, _, s, _ in rows], "Limit (data)": [r for *_, r in rows],
                        "OK?": ["✔" if o else '<b style="color:#bd0a0a">✘</b>' for o in ok]}))
    return [n for (n, *_), o in zip(rows, ok) if not o]


def verdict(violated, z, z_best, what, maximize=False, unit="$"):
    """Banner: feasible or not, value of the objective, comparison with the optimum."""
    money = lambda v: f"{unit}{v:,.2f}" if unit == "$" else f"{v:,.2f} {unit}"
    if violated:
        banner(False, f"<b>INFEASIBLE plan</b>: it violates {', '.join(violated)}.<br>{what} = {money(z)}, but this "
                      f"plan is not allowed. Change the decisions until every constraint shows ✔.", top=True)
        return
    gap = (z_best - z) if maximize else (z - z_best)
    tail = ("<b>This is the optimal plan.</b>" if abs(gap) < 1e-3 * max(1, abs(z_best)) else
            f"The best feasible plan reaches {money(z_best)}: you are {money(abs(gap))} "
            f"{'below' if maximize else 'above'} it. Can you do better?")
    banner(True, f"<b>Feasible plan</b>: {what} = {money(z)}.<br>{tail}", top=True)


def slider(desc, value, lo, hi, step):
    return W.FloatSlider(value=value, min=lo, max=hi, step=step, description=desc, continuous_update=False,
                         style={"description_width": "150px"}, layout=W.Layout(width="420px"))


def explorer(groups, view, best, chart_beside_controls=False):
    """groups: {"Column title": {name: widget}}; best: {name: optimal value}. Redraws on every change.
    chart_beside_controls: the chart goes next to the controls (e.g. a map to look at while ticking roads)."""
    controls = {k: w for g in groups.values() for k, w in g.items()}
    cols = [W.VBox([W.HTML(f"<b>{title}</b>")] + list(g.values())) for title, g in groups.items()]
    out, chart = W.HTML(), W.HTML()

    def refresh(_=None):
        PARTS.clear()
        try:
            view(**{k: w.value for k, w in controls.items()})
        except Exception:
            plt.close("all")
            PARTS.append(f"<pre style='color:#bd0a0a'>{traceback.format_exc()}</pre>")
        # layout: verdict banner on top, then tables (left) and chart (right) side by side, so it fits one screen
        banners = [h for h in PARTS if h.startswith("<div")]
        figures = [h for h in PARTS if h.startswith("<img")]
        tables = [h for h in PARTS if h not in banners and h not in figures]
        figures = [h.replace("<img ", '<img style="width:100%;max-width:520px;height:auto" ', 1) for h in figures]
        if chart_beside_controls:
            chart.value = f'<div style="width:480px;max-width:100%">{"".join(figures)}</div>'
            out.value = "".join(banners) + "".join(tables)
            return
        out.value = ("".join(banners) +
                     '<div style="display:flex;flex-wrap:nowrap;gap:18px;align-items:flex-start">'
                     f'<div style="flex:0 0 auto">{"".join(tables)}</div>'
                     f'<div style="flex:1 1 280px;min-width:240px">{"".join(figures)}</div></div>')

    def load(values):
        for k, w in controls.items():
            w.unobserve(refresh, names="value")
            w.value = values(k)
            w.observe(refresh, names="value")
        refresh()

    b_opt = W.Button(description="Load the optimal plan", button_style="success", layout=W.Layout(width="200px"))
    b_zero = W.Button(description="Reset to 0", layout=W.Layout(width="120px"))
    b_opt.on_click(lambda _: load(lambda k: best[k]))
    b_zero.on_click(lambda _: load(lambda k: False if isinstance(controls[k], W.Checkbox) else 0.0))
    for w in controls.values():
        w.observe(refresh, names="value")
    refresh()
    top = W.HBox(cols + [chart]) if chart_beside_controls else W.HBox(cols)
    display(W.VBox([top, W.HBox([b_opt, b_zero]), out]))


_ANS = json.loads(base64.b64decode("e30="))


def check(key, value, tol=0.01):
    """Compare your answer with the expected one (relative tolerance 1%)."""
    if value is None:
        print(f"⏳ {key}: not answered yet"); return
    exp = _ANS[key]
    ok = abs(float(value) - exp) <= tol * max(1.0, abs(exp))
    print(f"✅ {key}: correct" if ok else f"❌ {key}: not yet")


def _num(v):
    s = f"{round(float(v), 2):,.2f}".rstrip("0").rstrip(".")
    return "0" if s in ("-0", "") else s


def equations(symbols, values, objective, rows, z_best, what, maximize=False, unit="$"):
    """The model written with the slider values in place of the variables.
    symbols: HTML names of the variables; objective: its coefficients; rows: (name, coefficients, sense, limit).
    A violated constraint shows the relation that really holds, in red: ≤ becomes ≥ (and ≥ becomes ≤)."""
    tol = 1e-4
    x = np.asarray(values, float)
    var = lambda s: f'<i style="font-family:serif;font-size:15px">{s}</i>'
    val = lambda v: f'<b style="color:#1f5fbf">{_num(v)}</b>'

    def terms(coefs, plug):
        out = []
        for a, s, v in zip(coefs, symbols, x):
            if a == 0:
                continue
            k = "" if abs(a) == 1 else f"{abs(a):,g}"
            body = (f"{k}×{val(v)}" if k else val(v)) if plug else k + var(s)
            out.append((("−" if a < 0 else "") if not out else (" − " if a < 0 else " + ")) + body)
        return "".join(out)

    def single(coefs):                                   # e.g. x_T ≤ 20: no need to write "= value"
        nz = [a for a in coefs if a != 0]
        return len(nz) == 1 and abs(nz[0]) == 1

    z = float(np.dot(objective, x))
    head = "max" if maximize else "min"
    body = [("Objective", f"{head} {var('z')} = {terms(objective, False)}",
             f"{var('z')} = {terms(objective, True)} = <b>{_num(z)}</b>", what, False)]
    violated = []
    for name, coefs, sense, rhs in rows:
        lhs = float(np.dot(coefs, x))
        ok = lhs <= rhs + tol if sense == "≤" else lhs >= rhs - tol
        if ok:
            shown = f'<b style="color:#067429">{sense}</b>'
            check = "✔ at the limit" if abs(lhs - rhs) <= tol else f"✔ slack {_num(abs(rhs - lhs))}"
        else:
            violated.append(name)
            shown = f'<b style="color:#bd0a0a;font-size:18px">{"≥" if sense == "≤" else "≤"}</b>'
            check = (f'<b style="color:#bd0a0a">✘ {"over" if sense == "≤" else "short"} by {_num(abs(lhs - rhs))}'
                     f' (should be {sense})</b>')
        plan = terms(coefs, True) + ("" if single(coefs) else f" = <b>{_num(lhs)}</b>")
        body.append((name, f"{terms(coefs, False)} {sense} {_num(rhs)}", f"{plan} {shown} {_num(rhs)}", check, not ok))
    body.append(("Non-negativity", ", ".join(var(s) for s in symbols) + " ≥ 0",
                 ", ".join(f"{var(s)} = {val(v)}" for s, v in zip(symbols, x)) + " ≥ 0",
                 "✔" if (x >= -tol).all() else '<b style="color:#bd0a0a">✘</b>', False))

    td = 'style="padding:5px 12px;border-bottom:1px solid #ddd;text-align:left;white-space:nowrap"'
    html = ('<table style="border-collapse:collapse;margin:0 0 8px 0;font-size:14px">'
            f'<tr><th {td}></th><th {td}>With your plan (slider values)</th>'
            f'<th {td}>Check</th></tr>')
    for name, _, plan, check, bad in body:
        bg = ' style="background:#fff0f0"' if bad else ""
        html += f"<tr{bg}><td {td}><b>{name}</b></td><td {td}>{plan}</td><td {td}>{check}</td></tr>"
    PARTS.append(html + "</table>")
    verdict(violated, z, z_best, what, maximize=maximize, unit=unit)

print("Ready. Run each explorer cell below, then move the sliders.")


[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: pip install --upgrade pip


Ready. Run each explorer cell below, then move the sliders.


---
## 1 · The carpenter: what should she make this week?
A carpenter makes tables and chairs.

| | Profit (\$/unit) | Work (h/unit) | Material (units/unit) |
|---|---|---|---|
| Table | 500 | 2 | 1 |
| Chair | 300 | 1 | 2 |
| **Available each week** | | 40 | 50 |

**Try this**
* Make only tables, as many as possible. Which resource runs out? What is left of the other one?
* Make only chairs, as many as possible. Same questions.
* Find a better mix. In the best plan, how much work and material is left over?

In [2]:
# @title 🪑 Carpenter explorer (table profit $500): choose the plan, watch profit, hours and material
def _explorer():
    PROFIT = np.array([500, 300])                       # $ per table, per chair (fixed data)
    USE = np.array([[2, 1],                                # hours of work per table, per chair
                    [1, 2]])                               # units of material per table, per chair
    AVAILABLE = np.array([40, 50])
    RESOURCES = ["work (h)", "material (units)"]
    x_best, z_best = optimum(PROFIT, A_ub=USE, b_ub=AVAILABLE, bounds=[(0, None)] * 2, maximize=True)


    def carpenter_view(t, c):
        x = np.array([t, c])
        violated = constraints([(n, u, "≤", a) for n, u, a in zip(RESOURCES, USE @ x, AVAILABLE)])
        z = PROFIT @ x
        verdict(violated, z, z_best, "weekly profit", maximize=True)
        table(pd.DataFrame({"Product": ["Tables", "Chairs"], "Units": x, "Profit ($/unit)": PROFIT,
                            "Profit ($)": PROFIT * x}))


    explorer({"Your decisions: units made this week": dict(t=slider("t tables", 0, 0, 30, 0.5),
                                                            c=slider("c chairs", 0, 0, 30, 0.5))},
             carpenter_view, dict(zip(["t", "c"], x_best)))


_explorer()

### Formulation
**Variables:** $t$ = tables made, $c$ = chairs made this week.

$$
\begin{aligned}
\max\ & z = 500t + 300c && \text{(weekly profit)} \\
\text{s.t. } & 2t + c \le 40 && \text{(hours of work)} \\
& t + 2c \le 50 && \text{(units of material)} \\
& t, c \ge 0
\end{aligned}
$$

**The model with your numbers.** Same sliders as above, but now their values replace $t$ and $c$ in every equation of the formulation. A satisfied constraint keeps its sign ($\le$). When a constraint is violated, its sign **flips in red** ($\le$ becomes $\ge$): the equation itself tells you the plan is not allowed.

In [ ]:
# @title 🧮 Carpenter model with your numbers: the sliders replace t and c in every equation
def _explorer():
    PROFIT = [500, 300]                                    # $ per table, per chair (fixed data)
    USE = [[2, 1],                                         # hours of work per table, per chair
           [1, 2]]                                         # units of material per table, per chair
    AVAILABLE = [40, 50]
    RESOURCES = ["hours of work", "units of material"]
    x_best, z_best = optimum(PROFIT, A_ub=USE, b_ub=AVAILABLE, bounds=[(0, None)] * 2, maximize=True)


    def model_view(t, c):
        equations(["t", "c"], [t, c], PROFIT, [(n, u, "≤", a) for n, u, a in zip(RESOURCES, USE, AVAILABLE)],
                  z_best, "weekly profit", maximize=True)


    explorer({"Your decisions: units made this week": dict(t=slider("t tables", 0, 0, 30, 0.5),
                                                            c=slider("c chairs", 0, 0, 30, 0.5))},
             model_view, dict(zip(["t", "c"], x_best)))


_explorer()

**PuLP, version 1: write the model directly.** One variable per decision, and each line of the formulation
becomes one line of code.

In [3]:
import pulp

model = pulp.LpProblem("carpenter", pulp.LpMaximize)
t = model.add_variable("tables", lowBound=0)        # t ≥ 0
c = model.add_variable("chairs", lowBound=0)        # c ≥ 0

model += 500 * t + 300 * c                          # objective: weekly profit
model += 2 * t + c <= 40, "work"                    # hours of work
model += t + 2 * c <= 50, "material"                # units of material

result = model.solve(pulp.COIN_CMD(msg=False))
print(result.status_str, f"· profit = ${pulp.value(model.objective):,.0f}")
print(f"  tables = {t.value():g}, chairs = {c.value():g}")

Optimal · profit = $11,000
  tables = 10, chairs = 20


**PuLP, version 2: data and sums.** The data go in lists (one entry per product), the variables in a
dictionary, and each sum $\sum_j a_j x_j$ becomes `pulp.lpSum(...)`. The model no longer depends on the number
of products: adding a product means adding one entry to each list.

In [4]:
import pulp

products = ["Table", "Chair"]
profit = [500, 300]                                  # $ per unit
work = [2, 1]                                        # hours per unit
material = [1, 2]                                    # units of material per unit
hours, wood = 40, 50                                 # available each week
J = range(len(products))                             # j = 0, 1

model = pulp.LpProblem("carpenter", pulp.LpMaximize)
x = [model.add_variable(f"x_{p}", lowBound=0) for p in products]     # x[j] = units of product j

model += pulp.lpSum(profit[j] * x[j] for j in J)                       # Σ profit_j x_j
model += pulp.lpSum(work[j] * x[j] for j in J) <= hours, "work"        # Σ work_j x_j ≤ 40
model += pulp.lpSum(material[j] * x[j] for j in J) <= wood, "material"

result = model.solve(pulp.COIN_CMD(msg=False))
print(result.status_str, f"· profit = ${pulp.value(model.objective):,.0f}")
for j in J:
    print(f"  {products[j]:<6} {x[j].value():5.1f} units")

Optimal · profit = $11,000
  Table   10.0 units
  Chair   20.0 units


### Same question, new price
The market changes: a **table now earns only \$100** (chairs still earn \$300). Same workshop, same hours,
same material.

**Try this**
* What changes in the model: the constraints or the objective?
* Write the new model in PuLP below and solve it. Is the previous plan (10 tables, 20 chairs) still the best?
* Which resource is now left over?

In [ ]:
# ✍️ TODO: write the carpenter model with the new price (a table now earns $100) in PuLP,
#          solve it, and print the profit and the number of tables and chairs.
#          Tip: start from one of the PuLP cells above. Which line changes?


---
## 2 · The workshop grows: six products, three resources, a market limit
| | Table | Chair | Desk | Bookcase | Bed | Cabinet | Available |
|---|---|---|---|---|---|---|---|
| Profit (\$/unit) | 500 | 300 | 700 | 450 | **900** | 600 | |
| Carpentry (h) | 1 | 5 | 4 | 2 | 5 | 2 | 160 h |
| Finishing (h) | 1 | 4 | 2 | 2 | 3 | 1 | 100 h |
| Wood (units) | 2 | 3 | 2 | 2 | 3 | 4 | 180 units |

At most **20 tables** can be sold per week.

**Try this**
* By hand: make only the most profitable product (beds), as many as possible. Which resource stops you? What
  profit do you get?
* Write the model in PuLP below and solve it. How far was the "beds only" plan from the optimum?
* In the optimal plan, which product is **not** made at all? Which resources are used to the last unit?

### Formulation
**Variables:** $x_T, x_C, x_D, x_K, x_B, x_A$ = tables, chairs, desks, bookcases, beds and cabinets made this
week.

$$
\begin{aligned}
\max\ & z = 500x_T + 300x_C + 700x_D + 450x_K + 900x_B + 600x_A \\
\text{s.t. } & x_T + 5x_C + 4x_D + 2x_K + 5x_B + 2x_A \le 160 && \text{(carpentry)} \\
& x_T + 4x_C + 2x_D + 2x_K + 3x_B + x_A \le 100 && \text{(finishing)} \\
& 2x_T + 3x_C + 2x_D + 2x_K + 3x_B + 4x_A \le 180 && \text{(wood)} \\
& x_T \le 20 && \text{(market)} \\
& x_T, x_C, x_D, x_K, x_B, x_A \ge 0
\end{aligned}
$$

**General form:** $\max \sum_j p_j x_j$ s.t. $\sum_j a_{ij} x_j \le b_i$ for every resource $i$, $x_j \ge 0$.

**Your turn: write the model in PuLP.** The data are already in lists (one entry per product) and a matrix `use[i][j]` (resource $i$ per unit of product $j$). Write the variables, the objective, one constraint $\sum_j a_{ij} x_j \le b_i$ per resource with `pulp.lpSum`, and the market limit, then solve.

In [ ]:
import pulp

products = ["Table", "Chair", "Desk", "Bookcase", "Bed", "Cabinet"]
profit = [500, 300, 700, 450, 900, 600]              # $ per unit
resources = ["carpentry", "finishing", "wood"]
use = [[1, 5, 4, 2, 5, 2],                           # use[i][j]: resource i per unit of product j
       [1, 4, 2, 2, 3, 1],
       [2, 3, 2, 2, 3, 4]]
available = [160, 100, 180]
max_tables = 20

# ✍️ TODO: write the workshop model in PuLP (variables, objective, one constraint per resource, market limit),
#          solve it, and print the profit and the units of each product.


---
## 3 · Dorian Auto: advertise effectively but cheaply
Dorian Auto buys TV ads to reach at least **28 million women** and **24 million men**.

| Ad | Cost (\$/ad) | Women reached (millions) | Men reached (millions) |
|---|---|---|---|
| Comedy show | 50,000 | 7 | 2 |
| Football game | 100,000 | 2 | 12 |

This time we **minimize** a cost, and the constraints are **at least** ($\ge$).

**Try this**
* Buy only comedy ads, as few as possible. Which audience is reached last?
* Find the cheapest plan with the sliders. Is it a whole number of ads?
* Load the optimal plan, then round it to whole ads. Is the rounded plan still feasible?

In [10]:
# @title 📺 Dorian Auto explorer: choose the ads, watch cost and audience
def _explorer():
    COST = np.array([50_000, 100_000])                     # $ per comedy ad, per football ad (fixed data)
    REACH = np.array([[7, 2],                              # millions of women reached per ad
                      [2, 12]])                            # millions of men reached per ad
    NEED = np.array([28, 24])
    AUDIENCES = ["women (millions)", "men (millions)"]
    x_best, z_best = optimum(COST, A_ub=-REACH, b_ub=-NEED, bounds=[(0, None)] * 2)


    def dorian_view(x1, x2):
        x = np.array([x1, x2])
        violated = constraints([(n, r, "≥", d) for n, r, d in zip(AUDIENCES, REACH @ x, NEED)])
        verdict(violated, COST @ x, z_best, "advertising cost")
        table(pd.DataFrame({"Ad": ["Comedy", "Football"], "Ads bought": x, "Cost ($)": COST * x}))


    explorer({"Your decisions: ads bought": dict(x1=slider("x₁ comedy ads", 0, 0, 12, 0.1),
                                                  x2=slider("x₂ football ads", 0, 0, 15, 0.1))},
             dorian_view, dict(zip(["x1", "x2"], x_best)))


_explorer()

### Formulation
**Variables:** $x_1$ = comedy ads, $x_2$ = football ads.

$$
\begin{aligned}
\min\ & z = 50{,}000x_1 + 100{,}000x_2 && \text{(cost)} \\
\text{s.t. } & 7x_1 + 2x_2 \ge 28 && \text{(women, millions)} \\
& 2x_1 + 12x_2 \ge 24 && \text{(men, millions)} \\
& x_1, x_2 \ge 0
\end{aligned}
$$

**The model with your numbers.** Same sliders as above, but now their values replace $x_1$ and $x_2$ in every equation of the formulation. A satisfied constraint keeps its sign ($\ge$). When a constraint is violated, its sign **flips in red** ($\ge$ becomes $\le$): the equation itself tells you the plan is not allowed.

In [ ]:
# @title 🧮 Dorian Auto model with your numbers: the sliders replace x₁ and x₂ in every equation
def _explorer():
    COST = [50_000, 100_000]                               # $ per comedy ad, per football ad (fixed data)
    REACH = [[7, 2],                                       # millions of women reached per ad
             [2, 12]]                                      # millions of men reached per ad
    NEED = [28, 24]
    AUDIENCES = ["women (millions)", "men (millions)"]
    x_best, z_best = optimum(COST, A_ub=-np.array(REACH), b_ub=-np.array(NEED), bounds=[(0, None)] * 2)


    def model_view(x1, x2):
        equations(["x<sub>1</sub>", "x<sub>2</sub>"], [x1, x2], COST,
                  [(n, r, "≥", d) for n, r, d in zip(AUDIENCES, REACH, NEED)], z_best, "advertising cost")


    explorer({"Your decisions: ads bought": dict(x1=slider("x₁ comedy ads", 0, 0, 12, 0.1),
                                                  x2=slider("x₂ football ads", 0, 0, 15, 0.1))},
             model_view, dict(zip(["x1", "x2"], x_best)))


_explorer()

**PuLP, version 1: write the model directly.** Note `pulp.LpMinimize` and the `>=` constraints.

In [11]:
import pulp

model = pulp.LpProblem("dorian", pulp.LpMinimize)
x1 = model.add_variable("comedy_ads", lowBound=0)
x2 = model.add_variable("football_ads", lowBound=0)

model += 50_000 * x1 + 100_000 * x2                 # objective: cost
model += 7 * x1 + 2 * x2 >= 28, "women"             # millions of women reached
model += 2 * x1 + 12 * x2 >= 24, "men"              # millions of men reached

result = model.solve(pulp.COIN_CMD(msg=False))
print(result.status_str, f"· cost = ${pulp.value(model.objective):,.0f}")
print(f"  comedy ads = {x1.value():g}, football ads = {x2.value():g}")

Optimal · cost = $320,000
  comedy ads = 3.6, football ads = 1.4


**PuLP, version 2: data and sums.** One list per ad type, one row of `reach` per audience, and one loop for the
audience constraints $\sum_j r_{ij} x_j \ge d_i$.

In [12]:
import pulp

ads = ["Comedy", "Football"]
cost = [50_000, 100_000]                             # $ per ad
audiences = ["women", "men"]
reach = [[7, 2],                                     # reach[i][j]: millions of audience i per ad j
         [2, 12]]
need = [28, 24]                                      # millions, at least
I, J = range(len(audiences)), range(len(ads))

model = pulp.LpProblem("dorian", pulp.LpMinimize)
x = [model.add_variable(f"x_{a}", lowBound=0) for a in ads]

model += pulp.lpSum(cost[j] * x[j] for j in J)
for i in I:
    model += pulp.lpSum(reach[i][j] * x[j] for j in J) >= need[i], audiences[i]

result = model.solve(pulp.COIN_CMD(msg=False))
print(result.status_str, f"· cost = ${pulp.value(model.objective):,.0f}")
for j in J:
    print(f"  {ads[j]:<8} {x[j].value():5.2f} ads")

Optimal · cost = $320,000
  Comedy    3.60 ads
  Football  1.40 ads


### Making it an integer program
You cannot buy 3.6 ads. Adding **integrality** ($x_1, x_2$ whole numbers) drops the **divisibility**
assumption of LP; proportionality, additivity and certainty still hold. In PuLP, it is one word:
`cat="Integer"`. Compare the integer optimum with the rounded LP plan $(4, 1)$.

In [13]:
import pulp

model = pulp.LpProblem("dorian_integer", pulp.LpMinimize)
x1 = model.add_variable("comedy_ads", lowBound=0, cat="Integer")      # whole ads only
x2 = model.add_variable("football_ads", lowBound=0, cat="Integer")

model += 50_000 * x1 + 100_000 * x2
model += 7 * x1 + 2 * x2 >= 28, "women"
model += 2 * x1 + 12 * x2 >= 24, "men"

result = model.solve(pulp.COIN_CMD(msg=False))
print(result.status_str, f"· cost = ${pulp.value(model.objective):,.0f}")
print(f"  comedy ads = {x1.value():g}, football ads = {x2.value():g}")

r1, r2 = 4, 1                                        # the LP optimum (3.6, 1.4), rounded
print(f"rounded plan (4, 1): women {7 * r1 + 2 * r2} million (need 28), men {2 * r1 + 12 * r2} million (need 24)")

Optimal · cost = $400,000
  comedy ads = 6, football ads = 1
rounded plan (4, 1): women 30 million (need 28), men 20 million (need 24)


---
## 4 · When the solver says no: infeasible and unbounded models
The solver does not always return a plan. Run the cell and read the **status** of each model.
* **Infeasible:** at least 10 units of A, at least 8 units of B, but at most 15 units in total.
* **Unbounded:** $\max\ 3x_1 + 2x_2$ s.t. $x_1 - x_2 \ge 1$: nothing limits the profit.

Both usually reveal a **modeling error**: a requirement that cannot be met, or a missing limit (resources,
market demand).

In [14]:
import pulp

infeasible = pulp.LpProblem("production_gone_wrong", pulp.LpMaximize)
a = infeasible.add_variable("x_A", lowBound=0)
b = infeasible.add_variable("x_B", lowBound=0)
infeasible += a + b
infeasible += a >= 10, "at_least_10_A"
infeasible += b >= 8, "at_least_8_B"
infeasible += a + b <= 15, "at_most_15_in_total"
print("Infeasible example:", infeasible.solve(pulp.COIN_CMD(msg=False)).status_str)

unbounded = pulp.LpProblem("too_good_to_be_true", pulp.LpMaximize)
x1 = unbounded.add_variable("x1", lowBound=0)
x2 = unbounded.add_variable("x2", lowBound=0)
unbounded += 3 * x1 + 2 * x2
unbounded += x1 - x2 >= 1, "only_constraint"
print("Unbounded example: ", unbounded.solve(pulp.COIN_CMD(msg=False)).status_str)

Infeasible example: Infeasible
Unbounded example:  Unbounded


---
## What did you learn?
* A model has three parts: **decision variables** (what you control), an **objective** (what you maximize or
  minimize) and **constraints** (what limits you). The **data** never change while you decide.
* Changing a price changes only the **objective**, yet the best plan can change completely.
* Intuition (the most profitable product first) can miss the optimum by a lot: the solver checks every resource
  at once.
* In PuLP, writing the model **directly** is fine for a few variables; with **data and `lpSum`**, the same code
  works for any number of products, resources or audiences.
* Whole-number decisions make the model an **integer program**; rounding the LP solution can be infeasible.